# LAB 05 - Bài 13 và Bài 14

Chạy code cell **Bài 13** rồi đến cell **Bài 14**.
- **Bài 13:** so sánh max_length 50, 100, 200 bằng cùng một LSTM; báo cáo phần trăm token giữ lại, thời gian train, Accuracy và F1.
- **Bài 14:** tự cài đặt SimpleRNNCell và đối chiếu một timestep với `torch.nn.RNN`.

Bài 13 tải ACL IMDb từ Stanford ở lần chạy đầu (~84 MB), sau đó huấn luyện 3 mô hình. Có thể giảm `EPOCHS` để chạy nhanh hơn.

In [1]:
# ============================================================
# BÀI 13 - EXPERIMENT 2: PADDING VÀ TRUNCATION
# Thử max_length = 50, 100, 200 và ghi lại:
# % token được giữ, training time, accuracy, F1.
# Mỗi lần dùng cùng một kiến trúc LSTM và cùng các thiết lập.
# ============================================================

from pathlib import Path
import tarfile
import urllib.request
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# Tải ACL IMDb nếu chưa có trong runtime Colab.
DATA_URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
DATA_DIR = Path("/content/aclImdb")
ARCHIVE_PATH = Path("/content/aclImdb_v1.tar.gz")

if not (DATA_DIR / "train" / "pos").exists():
    if not ARCHIVE_PATH.exists():
        print("Đang tải ACL IMDb từ Stanford (~84 MB)...")
        urllib.request.urlretrieve(DATA_URL, ARCHIVE_PATH)
    print("Đang giải nén dataset...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as archive:
        archive.extractall(path="/content")

# Đọc nội dung theo train/test và nhãn neg=0, pos=1.
def read_split(split):
    texts, labels = [], []
    for label_name, label_id in (("neg", 0), ("pos", 1)):
        folder = DATA_DIR / split / label_name
        for file_path in sorted(folder.glob("*.txt")):
            texts.append(file_path.read_text(encoding="utf-8", errors="replace").strip())
            labels.append(label_id)
    return texts, np.asarray(labels, dtype=np.int32)

train_texts, y_all = read_split("train")
test_texts, y_test = read_split("test")

# Giữ test riêng; vocabulary chỉ học từ train.
MAX_VOCAB_SIZE = 20_000
EMBEDDING_DIM = 64
BATCH_SIZE = 64
EPOCHS = 3
MAX_LENGTHS = [50, 100, 200]

X_train_texts, X_val_texts, y_train, y_val = train_test_split(
    train_texts,
    y_all,
    test_size=0.2,
    random_state=42,
    stratify=y_all,
)

tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_texts)
train_seq = tokenizer.texts_to_sequences(X_train_texts)
val_seq = tokenizer.texts_to_sequences(X_val_texts)
test_seq = tokenizer.texts_to_sequences(test_texts)

# % tokens được giữ tính trên toàn bộ train sequences trước padding.
total_train_tokens = sum(len(seq) for seq in train_seq)
if total_train_tokens == 0:
    raise ValueError("Không có token nào trong tập train.")

vocab_size = min(MAX_VOCAB_SIZE, len(tokenizer.word_index) + 1)
results = []

for max_len in MAX_LENGTHS:
    print(f"\n--- max_length = {max_len} ---")

    retained_tokens = sum(min(len(seq), max_len) for seq in train_seq)
    retained_percent = 100 * retained_tokens / total_train_tokens

    X_train = pad_sequences(train_seq, maxlen=max_len, padding="post", truncating="post")
    X_val = pad_sequences(val_seq, maxlen=max_len, padding="post", truncating="post")
    X_test = pad_sequences(test_seq, maxlen=max_len, padding="post", truncating="post")

    # Reset seed để điều kiện khởi tạo gần giống nhau giữa các lần thử.
    tf.keras.utils.set_random_seed(42)
    model = models.Sequential([
        layers.Input(shape=(max_len,)),
        layers.Embedding(vocab_size, EMBEDDING_DIM, mask_zero=True),
        layers.LSTM(32),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

    start_time = time.perf_counter()
    model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=2,
    )
    training_time = time.perf_counter() - start_time

    probabilities = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0).ravel()
    predictions = (probabilities >= 0.5).astype(np.int32)

    results.append({
        "max_length": max_len,
        "tokens_retained_percent": retained_percent,
        "training_time_seconds": training_time,
        "accuracy": accuracy_score(y_test, predictions),
        "f1": f1_score(y_test, predictions),
    })

results_df = pd.DataFrame(results)
print("\nBảng so sánh Bài 13:")
print(results_df.round(4).to_string(index=False))

# Gợi ý phân tích sau khi chạy:
# - max_length lớn hơn không đảm bảo Accuracy/F1 luôn cao hơn.
# - max_length nhỏ có thể cắt mất phủ định, từ chỉ cảm xúc hoặc phần kết luận.
# - So sánh cả thời gian chạy và % token giữ lại với Accuracy/F1.


Đang tải ACL IMDb từ Stanford (~84 MB)...
Đang giải nén dataset...


/tmp/ipykernel_5047/1932195655.py:32: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(path="/content")



--- max_length = 50 ---
Epoch 1/3
313/313 - 18s - 58ms/step - accuracy: 0.7093 - loss: 0.5470 - val_accuracy: 0.7700 - val_loss: 0.4758
Epoch 2/3
313/313 - 22s - 71ms/step - accuracy: 0.8562 - loss: 0.3438 - val_accuracy: 0.7698 - val_loss: 0.5112
Epoch 3/3
313/313 - 17s - 53ms/step - accuracy: 0.9078 - loss: 0.2431 - val_accuracy: 0.7596 - val_loss: 0.6234

--- max_length = 100 ---
Epoch 1/3
313/313 - 32s - 102ms/step - accuracy: 0.7487 - loss: 0.4910 - val_accuracy: 0.7920 - val_loss: 0.4523
Epoch 2/3
313/313 - 26s - 85ms/step - accuracy: 0.8953 - loss: 0.2705 - val_accuracy: 0.8116 - val_loss: 0.5003
Epoch 3/3
313/313 - 28s - 89ms/step - accuracy: 0.9287 - loss: 0.1979 - val_accuracy: 0.7966 - val_loss: 0.7810

--- max_length = 200 ---
Epoch 1/3
313/313 - 53s - 170ms/step - accuracy: 0.7752 - loss: 0.4666 - val_accuracy: 0.8428 - val_loss: 0.3621
Epoch 2/3
313/313 - 81s - 260ms/step - accuracy: 0.9100 - loss: 0.2416 - val_accuracy: 0.8544 - val_loss: 0.4262
Epoch 3/3
313/313 - 51s 

In [2]:
# ============================================================
# BÀI 14 - CORE IMPLEMENTATION: SIMPLE RNN CELL
# Cài đặt h_t = tanh(W_xh x_t + W_hh h_(t-1) + b_h)
# rồi đối chiếu với torch.nn.RNN ở đúng một timestep.
# ============================================================

import math
import torch
import torch.nn as nn

class SimpleRNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        # W_xh: (hidden_size, input_size)
        # W_hh: (hidden_size, hidden_size)
        self.W_xh = nn.Parameter(torch.empty(hidden_size, input_size))
        self.W_hh = nn.Parameter(torch.empty(hidden_size, hidden_size))
        self.b_h = nn.Parameter(torch.empty(hidden_size))
        self.reset_parameters()

    def reset_parameters(self):
        bound = 1 / math.sqrt(self.hidden_size)
        nn.init.uniform_(self.W_xh, -bound, bound)
        nn.init.uniform_(self.W_hh, -bound, bound)
        nn.init.uniform_(self.b_h, -bound, bound)

    def forward(self, x_t, h_prev):
        # x_t: (batch_size, input_size)
        # h_prev: (batch_size, hidden_size)
        a_t = x_t @ self.W_xh.T + h_prev @ self.W_hh.T + self.b_h
        h_t = torch.tanh(a_t)
        return h_t

# Test bằng một input nhỏ.
torch.manual_seed(7)
input_size = 3
hidden_size = 2
cell = SimpleRNNCell(input_size, hidden_size)

x_t = torch.tensor([[1.0, 0.5, -0.5]])
h_prev = torch.tensor([[0.5, 0.2]])
h_custom = cell(x_t, h_prev)
print("SimpleRNNCell output:", h_custom.detach().numpy())

# PyTorch nn.RNN xử lý chuỗi theo dạng (batch, seq_len, input_size)
# khi batch_first=True. Đặt trọng số giống custom cell để so sánh.
rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size,
              nonlinearity="tanh", batch_first=True, bias=True)
with torch.no_grad():
    rnn.weight_ih_l0.copy_(cell.W_xh)
    rnn.weight_hh_l0.copy_(cell.W_hh)
    rnn.bias_ih_l0.copy_(cell.b_h)
    rnn.bias_hh_l0.zero_()

x_sequence = x_t.unsqueeze(1)       # (batch=1, seq_len=1, input_size)
h0 = h_prev.unsqueeze(0)            # (num_layers=1, batch=1, hidden_size)
output, h_n = rnn(x_sequence, h0)
h_framework = h_n.squeeze(0)

print("PyTorch nn.RNN output:", h_framework.detach().numpy())
print("Sai khác tuyệt đối lớn nhất:", (h_custom - h_framework).abs().max().item())
assert torch.allclose(h_custom, h_framework, atol=1e-6), "Hai kết quả chưa khớp."
print("Đối chiếu thành công: kết quả custom cell khớp torch.nn.RNN.")

# Computation graph của cell:
# x_t --W_xh--┐
#             + --> tanh --> h_t
# h_prev-W_hh-┘


SimpleRNNCell output: [[0.05531239 0.16064717]]
PyTorch nn.RNN output: [[0.05531239 0.16064715]]
Sai khác tuyệt đối lớn nhất: 1.4901161193847656e-08
Đối chiếu thành công: kết quả custom cell khớp torch.nn.RNN.
